# FMT-350: vilka MARG-genretermer är mappade till MARC-enums - eller något annat i KBV?
https://kbse.atlassian.net/browse/FMT-350

In [ ]:
import pandas
import requests

## Vad är MARC enums och enum types i KBV?

Enumvärden (enum values) är värden som ofta kan användas i flera kodlistor (enum types). Värdena består av ett namn i klartext (enum value), en kod bestående av en bokstav eller siffra (code) som är det man anger i MARC:s fasta fält, och en svensk samt en engelsk etikett (label). Kodlistorna motsvarar olika positioner i MARC. Bokstavskoderna är inte unika, utan kan betyda olika saker i olika termlistor.

Av de 764 unika enum-värdena är 113 mappade, via exactMatch, till en marcgt-kod.

Läs mer om termerna här: https://id.loc.gov/vocabulary/marcgt.html
Läs mer om hur termerna/koderna används här: https://www.loc.gov/standards/sourcelist/genre-form.html

### Exempel

Värdet marc:LegalArticle, alltså koden "g", kan ingå i föjande kodlistor:
- marc:SerialsNatureType
- marc:BooksContentsType	
- marc:SerialsContentsType


In [ ]:
df = pandas.read_csv('../../marc_enums.tsv', sep='\t')
df = df.dropna(axis=1, how='all')
df.info()
df.head(5)

In [ ]:
df[["enumvalue"]].value_counts().head(5)

### Unika MARC-enums

In [ ]:
# Unika värden
unique_values = df.dropna(subset=["code"])
unique_values.info()
unique_values.head(5)

In [ ]:
# Finns det några enums som är deprecated?
unique_values["deprecated"].value_counts()

In [ ]:
# Vilka koder används i flest termlistor ?
unique_values["code"].value_counts().head(5)

### MARC-enums som har en MARC Genre Term som exactMatch

In [ ]:
# Mapped values
mapped = unique_values.loc[unique_values["exactMatch"].notna()]
mapped.info()
mapped.head(5)

### MARC-enums som inte har en MARC Genre Term som exactMatch

In [ ]:
# Unmapped values
unmapped = unique_values.loc[unique_values["exactMatch"].isna()]
unmapped.info()
unmapped.head(5)

In [ ]:
marcgts = requests.get("https://id.loc.gov/vocabulary/marcgt", headers={"Accept": "application/json"}).json()

In [ ]:
for i in marcgts:
    if "http://www.loc.gov/mads/rdf/v1#Authority" in i["@type"]:
        print(i["@id"][-3:], "\t", i.get("http://www.loc.gov/mads/rdf/v1#authoritativeLabel")[0].get("@value"))



In [ ]:
codes = [f'marcgt:{i["@id"][-3:]}' for i in marcgts if "http://www.loc.gov/mads/rdf/v1#Authority" in i["@type"]]
print(len(codes))
print(codes)

In [ ]:
mapped_codes = []
unmapped_codes = []
for c in codes:
    if c in unique_values["exactMatch"].values:
        mapped_codes.append(c)
    else:
        unmapped_codes.append(c)

print("Number of codes mapped in construct_enums:", len(mapped_codes))

print("Number of codes not mapped in construct_enums:", len(unmapped_codes))

print("\nUnmapped codes:", unmapped_codes)

**Finns i source/genreforms/genreforms.ttl (closeMatch)**
- marcgt:art
- marcgt:map
- marcgt:rem
- marcgt:boo
- marcgt:iss
- marcgt:scr
- marcgt:mod

**Finns i source/genreforms/rdacategories.ttl (closeMatch)**
- marcgt:nos

**Finns i source/genreforms/contentgenres.ttl (closeMatch)**
- marcgt:jou
- marcgt:fin